# R13 · Lymph node 5k at 100 to 2,000 genes: d_B vs d_W and the block structure

In lymph node 5k (top 400 of 4,624 genes) d_B barely follows the whole-matrix distance d_W (Fig S10, ρ = 0.19): its
genes co-vary across blocks. Is that the choice of the 400 most variable genes? The production layout is rebuilt at
the top 100, 200, 400, 800, 1,600 and 2,000 genes, the 1,000 random same-niche tile pairs of Fig S10 are redrawn on
each, and one example tile pair's matrices are kept at 100–1,600 genes (Fig S11).

The production build does not fit at 2,000 genes (~110 GB of covariances, ~220 GB of ultrametrics, a ~110 GB PCA
input), so `layout_stream.py` runs the same steps streamed: covariances recomputed from the counts when needed, the
ultrametric features in a float32 file on disk, sklearn's randomized PCA streamed over it, then Leiden, consensus
orders and blocks with `spindle_dev`'s own functions. **Check:** the same code at 400 genes against the cached layout
and R12's pairs (identical).

* **Prerequisites:** `R0_index_cache` (lymph node 5k: staged covariances, seed-0 layout), `R12_block_distance`.
* **Writes:** `results/block_distance_genes/genes_pairs.csv` (one row per tile pair and gene count), `genes_layout.csv`
  (per gene count: niches, blocks, entries inside blocks), `scale_examples.npz` (Fig S11's matrices); layouts in
  `results/_cache/lymph_node_5k/genes<p>/`. Existing gene counts are skipped.
* **Runtime / memory:** ~1.5 h on 16 cores from scratch; peak ~62 GB RAM at 2,000 genes, ~56 GB scratch on disk
  (deleted after each layout).
  `sbatch --cpus-per-task=16 --mem=96G --time=12:00:00 --job-name=R13_block_distance_genes --output=results/_logs/R13_block_distance_genes/%x_%j.out experiments/common/run_nb.sbatch experiments/compute/R13_block_distance_genes.ipynb`

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import adjusted_rand_score

import data
import layout_stream as ls
import results_io as rio

TOPIC = "block_distance_genes"
KEY = "lymph_node_5k"
GENES = (100, 200, 400, 800, 1600, 2000)
FIG_GENES = (100, 200, 400, 800, 1600)  # Fig S11's matrices
FORCE = False
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
OUT = rio.topic(TOPIC)
print(data.stamp(), f"{N_JOBS} cores")

{'tag': 'method-v2', 'describe': 'method-v3-4-gb026ef3-dirty', 'commit': 'b026ef3'} 16 cores


## Layouts and pairs per gene count (new gene counts only)

In [2]:
def check_400(sec, lay, pr):
    """The streamed build at 400 genes against the cache: genes, covariances, niches, gene orders, blocks, R12's pairs."""
    c = data.load_cache(KEY, data.EXAMPLE_SEED, parts=("layout",))
    assert (sec["genes"] == c.genes).all() and (sec["train"] == c.train).all()
    covs = data.staged_covs(KEY)
    for t in sec["train"][[0, len(sec["train"]) // 2, -1]]:
        assert np.array_equal(ls.tile_cov(sec["X"][sec["idx"][t]]), np.asarray(covs[int(t)]["cov"])), t
    L = c.layout
    ari = adjusted_rand_score(L.labels, lay["labels"])
    same = [k for k in set(lay["labels"].tolist())
            if k in L.block_dict and np.array_equal(np.flatnonzero(L.labels == k), np.flatnonzero(lay["labels"] == k))
            and np.array_equal(L.perm_list[k], lay["perm"][k]) and list(map(tuple, L.block_dict[k])) == list(map(tuple, lay["runs"][k]))]
    print(f"labels ARI {ari:.4f}; niches identical (tiles, order, blocks): {len(same)} of {len(set(lay['labels'].tolist()))}")
    r12 = pd.read_csv(rio.topic("block_distance") / "dataset_pairs.csv").query("key == @KEY")
    m = pr.merge(r12, on=["niche", "tile_a", "tile_b"], suffixes=("", "_r12"))
    print(f"pairs as R12: {len(m)} of {len(pr)}; max rel. diff d_W {np.max(np.abs(m.d_whole / m.d_whole_r12 - 1)):.1e}, "
          f"d_B {np.max(np.abs(m.d_block / m.d_block_r12 - 1)):.1e}")
    assert ari == 1.0 and len(same) == len(set(lay["labels"].tolist()))
    assert len(m) == len(pr) and np.allclose(m.d_whole, m.d_whole_r12, rtol=1e-4) and np.allclose(m.d_block, m.d_block_r12, rtol=1e-4)


def layout_row(n, lay):
    p, labels = lay["p"], lay["labels"]
    inside = sum(np.sum(labels == k) * sum((e - s) * (e - s + 1) / 2 for s, e in lay["runs"][k]) for k in lay["runs"])
    return {"key": KEY, "n_genes": n, "p": p, "tiles": len(labels), "niches": len(lay["runs"]),
            "blocks": int(sum(len(r) for r in lay["runs"].values())), "resolution": lay["resolution"],
            "entries_inside_blocks": inside / (len(labels) * p * (p + 1) / 2)}


old_p = pd.read_csv(OUT / "genes_pairs.csv") if (OUT / "genes_pairs.csv").exists() and not FORCE else None
old_l = pd.read_csv(OUT / "genes_layout.csv") if (OUT / "genes_layout.csv").exists() and not FORCE else None
done = set(old_p.n_genes) if old_p is not None else set()
new_p, new_l = [], []
for n in GENES:
    if n in done:
        continue
    sec = ls.section(KEY, n)
    lay = ls.layout(sec, data.CACHE / KEY / f"genes{n}", n_jobs=N_JOBS)
    pr = ls.pairs(sec, lay, n_jobs=N_JOBS)
    if n == 400:
        check_400(sec, lay, pr)
    new_p.append(pr.assign(key=KEY, n_genes=n)); new_l.append(layout_row(n, lay))
    print(f"{n} genes: {lay['p']} genes, {len(lay['runs'])} niches, {len(pr)} pairs", flush=True)
    del sec

[2026-11-11 21:45:13,286] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)


[     11 s] features: 6972 x 4950 (0.1 GB)
[     14 s] PCA: 30 components
[     31 s] Leiden resolution 1.20: 14 niches, sizes [984, 828, 709, 694, 604, 588, 547, 538, 401, 354, 285, 169, 165, 106]
[     31 s] consensus gene orders


[2026-11-11 21:45:46,360] INFO spindle_dev.layout: Niche 0: 4 blocks
[2026-11-11 21:45:46,377] INFO spindle_dev.layout: Niche 1: 3 blocks
[2026-11-11 21:45:46,392] INFO spindle_dev.layout: Niche 2: 4 blocks
[2026-11-11 21:45:46,400] INFO spindle_dev.layout: Niche 3: 4 blocks
[2026-11-11 21:45:46,408] INFO spindle_dev.layout: Niche 4: 4 blocks
[2026-11-11 21:45:46,416] INFO spindle_dev.layout: Niche 5: 3 blocks
[2026-11-11 21:45:46,423] INFO spindle_dev.layout: Niche 6: 3 blocks
[2026-11-11 21:45:46,431] INFO spindle_dev.layout: Niche 7: 2 blocks
[2026-11-11 21:45:46,439] INFO spindle_dev.layout: Niche 8: 4 blocks
[2026-11-11 21:45:46,447] INFO spindle_dev.layout: Niche 9: 4 blocks
[2026-11-11 21:45:46,455] INFO spindle_dev.layout: Niche 10: 4 blocks
[2026-11-11 21:45:46,463] INFO spindle_dev.layout: Niche 11: 3 blocks
[2026-11-11 21:45:46,472] INFO spindle_dev.layout: Niche 12: 4 blocks
[2026-11-11 21:45:46,480] INFO spindle_dev.layout: Niche 13: 3 blocks


[     32 s] blocks: 49 over 14 niches
100 genes: 100 genes, 14 niches, 1000 pairs


[2026-11-11 21:45:47,988] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)


[      5 s] features: 6972 x 19900 (0.6 GB)
[     10 s] PCA: 30 components
[     24 s] Leiden resolution 1.10: 13 niches, sizes [922, 907, 731, 728, 579, 544, 505, 484, 427, 364, 338, 282, 161]
[     25 s] consensus gene orders


... (6 outputs left out)

[2026-11-11 21:51:45,990] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)


[    474 s] features: 6972 x 1279200 (35.7 GB)
[   1351 s] PCA: 30 components
[   1366 s] Leiden resolution 1.10: 12 niches, sizes [978, 950, 864, 740, 664, 626, 601, 585, 538, 241, 93, 92]
[   1408 s] consensus gene orders


[2026-11-11 22:17:58,946] INFO spindle_dev.layout: Niche 0: 100 blocks
[2026-11-11 22:18:00,511] INFO spindle_dev.layout: Niche 1: 101 blocks
[2026-11-11 22:18:02,110] INFO spindle_dev.layout: Niche 2: 102 blocks
[2026-11-11 22:18:03,509] INFO spindle_dev.layout: Niche 3: 96 blocks
[2026-11-11 22:18:04,984] INFO spindle_dev.layout: Niche 4: 98 blocks
[2026-11-11 22:18:06,355] INFO spindle_dev.layout: Niche 5: 96 blocks
[2026-11-11 22:18:07,964] INFO spindle_dev.layout: Niche 6: 102 blocks
[2026-11-11 22:18:09,493] INFO spindle_dev.layout: Niche 7: 101 blocks
[2026-11-11 22:18:10,947] INFO spindle_dev.layout: Niche 8: 100 blocks
[2026-11-11 22:18:12,147] INFO spindle_dev.layout: Niche 9: 96 blocks
[2026-11-11 22:18:13,569] INFO spindle_dev.layout: Niche 10: 98 blocks
[2026-11-11 22:18:15,148] INFO spindle_dev.layout: Niche 11: 100 blocks


[   1587 s] blocks: 1190 over 12 niches
1600 genes: 1600 genes, 12 niches, 1000 pairs


## Write

In [3]:
if new_p:
    tag = data.stamp()["tag"]
    gp = pd.concat(([old_p] if old_p is not None else []) + [d.assign(tag=tag) for d in new_p], ignore_index=True)
    gl = pd.concat(([old_l] if old_l is not None else []) + [pd.DataFrame(new_l).assign(tag=tag)], ignore_index=True)
    rio.write_csv(gp.sort_values("n_genes", kind="stable"), TOPIC, "genes_pairs.csv",
                  "lymph node 5k, layout rebuilt at the top 100-2,000 genes (layout_stream.py, seed-0 split): 1,000 "
                  "random same-niche pairs of index tiles per gene count (default_rng(0)); d_whole, d_block, the split "
                  "of d_whole^2 into entries inside (d_on) and between (d_off) blocks, tile a's mean |r| within and "
                  "between blocks (gene pairs with counts in the tile)")
    rio.write_csv(gl.sort_values("n_genes"), TOPIC, "genes_layout.csv",
                  "per gene count: index tiles, niches, blocks, Leiden resolution, share of the upper-triangle "
                  "entries inside blocks (tile-weighted)")
gp = pd.read_csv(OUT / "genes_pairs.csv")
pd.read_csv(OUT / "genes_layout.csv")

wrote results/block_distance_genes/genes_pairs.csv: 6000 rows
wrote results/block_distance_genes/genes_layout.csv: 6 rows


,key,n_genes,p,tiles,niches,blocks,resolution,entries_inside_blocks,tag
0,lymph_node_5k,100,100,6972,14,49,1.2,0.383574,method-v2
1,lymph_node_5k,200,200,6972,13,120,1.1,0.139104,method-v2
2,lymph_node_5k,400,400,6972,15,302,1.3,0.062581,method-v2
3,lymph_node_5k,800,800,6972,14,642,1.2,0.026238,method-v2
4,lymph_node_5k,1600,1600,6972,12,1190,1.1,0.011350,method-v2
5,lymph_node_5k,2000,2000,6972,12,1494,1.1,0.009005,method-v2


## Summary (shown, not saved)

In [4]:
def summary(g):
    cv = lambda x: x.std() / x.mean()  # noqa: E731
    return pd.Series({"spearman_rho": spearmanr(g.d_whole, g.d_block)[0], "pearson_r": pearsonr(g.d_whole, g.d_block)[0],
                      "median_gap": np.median(g.d_whole - g.d_block),
                      "off2_share": np.median(g.d_off ** 2 / g.d_whole ** 2),
                      "cv_d_on": cv(g.d_on), "cv_d_off": cv(g.d_off),
                      "rho_on_block": spearmanr(g.d_on, g.d_block)[0],
                      "abs_r_within": g.abs_r_within_a.median(), "abs_r_between": g.abs_r_between_a.median()})

gp.groupby("n_genes").apply(summary).T.round(3)

n_genes,100,200,400,800,1600,2000
spearman_rho,0.721,0.735,0.192,-0.168,-0.197,-0.254
pearson_r,0.919,0.824,0.509,0.018,0.147,0.621
median_gap,0.311,0.725,0.874,0.837,0.611,0.524
off2_share,0.429,0.735,0.856,0.916,0.941,0.946
cv_d_on,0.207,0.140,0.224,0.279,0.404,0.569
cv_d_off,0.147,0.045,0.044,0.058,0.057,0.059
rho_on_block,0.985,0.874,0.769,0.660,0.673,0.646
abs_r_within,0.121,0.124,0.121,0.115,0.103,0.097
abs_r_between,0.077,0.085,0.084,0.080,0.076,0.073


## Example matrices (Fig S11)
One example tile T and a partner T′ in the same niche at every gene count of `FIG_GENES`: T is the tile with the median
cell count in the largest niche at 400 genes (the production layout); T′ is drawn at random (`default_rng(0)`) from
the tiles that share T's niche at every gene count. Per gene count, in the gene order of T's niche: the niche's mean
correlation, T's correlation and |log C_T − log C_T′| (whose squared sum is p·d_W²), with the niche's blocks.

In [5]:
import json
import pickle

if FORCE or not (OUT / "scale_examples.npz").exists():
    lays = {n: pickle.loads((data.CACHE / KEY / f"genes{n}" / "layout.pkl").read_bytes()) for n in FIG_GENES}
    train = lays[400]["train"]
    n_cells = np.asarray(json.loads((data.CACHE / KEY / "covs.json").read_text())["n"])[train]
    lab400 = lays[400]["labels"]
    big = np.bincount(lab400).argmax()
    mem = np.flatnonzero(lab400 == big)
    ia = mem[np.argsort(n_cells[mem], kind="stable")[len(mem) // 2]]
    same = np.ones(len(train), bool)
    for n, L in lays.items():
        same &= L["labels"] == L["labels"][ia]
    same[ia] = False
    ib = np.random.default_rng(0).choice(np.flatnonzero(same))
    ta, tb = int(train[ia]), int(train[ib])
    print(f"T = tile {ta} ({n_cells[ia]} cells), T' = tile {tb} ({n_cells[ib]} cells); "
          f"{same.sum()} tiles share T's niche at every gene count")
    out = {"tile_a": ta, "tile_b": tb, "genes": np.array(FIG_GENES)}
    for n in FIG_GENES:
        sec = ls.section(KEY, n)
        ex = ls.example(sec, lays[n], ta, tb, n_jobs=N_JOBS)
        for name, v in ex.items():
            v = np.asarray(v)
            out[f"p{n}_{name}"] = v.astype(np.float16) if v.dtype == np.float32 else v
        print(f"{n} genes: niche {ex['niche']} ({ex['niche_tiles']} tiles, {len(ex['runs'])} blocks)", flush=True)
        del sec
    np.savez_compressed(OUT / "scale_examples.npz", **out)
    rio.register(TOPIC, OUT / "scale_examples.npz",
                 "Fig S11: tiles T, T' (prepare() order) and, per gene count p in genes, T's niche (p{p}_niche, "
                 "_niche_tiles), its gene order (_perm, _genes) and blocks (_runs), and in that order the niche's mean "
                 "correlation, T's correlation and |log C_T - log C_T'| (float16)")
z = np.load(OUT / "scale_examples.npz")
print({n: (int(z[f"p{n}_niche"]), int(z[f"p{n}_niche_tiles"]), len(z[f"p{n}_runs"])) for n in z["genes"]})

T = tile 466 (98 cells), T' = tile 7056 (88 cells); 3 tiles share T's niche at every gene count


[2026-11-11 22:23:54,534] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)


100 genes: niche 10 (285 tiles, 4 blocks)


[2026-11-11 22:23:57,594] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)


200 genes: niche 3 (728 tiles, 10 blocks)


[2026-11-11 22:24:01,041] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)


400 genes: niche 0 (873 tiles, 20 blocks)


[2026-11-11 22:24:05,274] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)


800 genes: niche 5 (618 tiles, 47 blocks)


[2026-11-11 22:24:11,641] INFO spindle_dev.preprocessing: build_quadtree_tiles: dropping 366 low-density tile(s) (19750 spots)


1600 genes: niche 3 (740 tiles, 96 blocks)
{np.int64(100): (10, 285, 4), np.int64(200): (3, 728, 10), np.int64(400): (0, 873, 20), np.int64(800): (5, 618, 47), np.int64(1600): (3, 740, 96)}


In [6]:
rio.write_manifest(TOPIC, "R13_block_distance_genes.ipynb",
                   inputs=["the lymph node 5k section (datasets.yaml)", "results/_cache/lymph_node_5k/covs.npy (R0, check)",
                           "results/_cache/lymph_node_5k/seed0/layout.pkl (R0, check)",
                           "results/block_distance/dataset_pairs.csv (R12, check)", "results/_cache/lymph_node_5k/covs.json (cell counts)"])

wrote results/block_distance_genes/manifest.json: 3 files; peak memory 40.94 GB


{'topic': 'block_distance_genes',
 'notebook': 'R13_block_distance_genes.ipynb',
 'date': '2026-11-11 22:24',
 'tag': 'method-v2',
 'describe': 'method-v3-4-gb026ef3-dirty',
 'commit': 'b026ef3',
 'inputs': ['the lymph node 5k section (datasets.yaml)',
  'results/_cache/lymph_node_5k/covs.npy (R0, check)',
  'results/_cache/lymph_node_5k/seed0/layout.pkl (R0, check)',
  'results/block_distance/dataset_pairs.csv (R12, check)',
  'results/_cache/lymph_node_5k/covs.json (cell counts)'],
 'peak_rss_gb': 40.94,
 'notes': '',
 'files': {'genes_layout.csv': {'description': 'per gene count: index tiles, niches, blocks, Leiden resolution, share of the upper-triangle entries inside blocks (tile-weighted)',
   'rows': 6,
   'bytes': 514},
  'genes_pairs.csv': {'description': "lymph node 5k, layout rebuilt at the top 100-2,000 genes (layout_stream.py, seed-0 split): 1,000 random same-niche pairs of index tiles per gene count (default_rng(0)); d_whole, d_block, the split of d_whole^2 into entries i